In [0]:
# Databricks notebook source
from typing import Any

# COMMAND ----------

import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

# COMMAND ----------

dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()



In [0]:

# COMMAND ----------

def load_curated_auction_real_estate_thesis(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "AuctionRealEstateId String, OriginSourceSystem String, SourceSystem String,IngestionTimestamp timestamp, BatchId String,"
        #     "AuctionRealEstateKey String, AuctionRealEstateCode String, AuctionCode String, AuctionObjKey string, RealEstateGroupCode int,"
        #     "DebtorPartyKey String, OriginDebtorTaxRegistrationNumber String, OriginDebtorDescription String, AuctionDate Date, AuctionPublishedDate date, AuctionPostedDate date,"
        #     "PartyKey string, RealEstateKey string, RealEstateOwnershipKey string, CaseCode int,"
        #     "AuctionFirstOfferPrice String, SeizureReportNumber String, SeizureReportYear int, HastenerCode String, Hastener String, HastenerTaxRegistrationNumber String,"
        #     "AuctionFinalPrice String, AuctionStatusCode String, AuctionStatus String, AuctioneerName String, AuctioneerEmail String, AuctioneerPhone String, TitleGR String,"
        #     "Area String, District String, Municipality String, Prefecture String, GeoCoordinatesX String, GeoCoordinatesY String, GeoLevel String, Address String,"
        #     "PostalCode int, Category String, SubCategory String, LandeaUrl String, FloorRawValue String, ConstructionYear int, SurfaceSqm float, Tags String,"
        #     "TagValues String, ActiveFlag boolean, TNMergeCode String, LandRegistration String, BidderCode String, DateEntered String, EnteredBy String, AuctionOwnershipTypeCode int,"
        #     "AuctionOwnershipType String, FirstBidder String, LawFirm String,FileDate date,FileName string, BusinessDate date"
        # )


        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.AuctionRealEstate"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                    spark,
                    curated_table_path,
                    run_date,  # type: ignore
                    "thesis",
                )        
        else:
            logging.info("Tables doe not exist, no need to delete entries for business date!")

        # ================== prlinked_properties ==================

        prlinked_properties_extra_filters = None

        if prlinked_properties_extra_filters is not None:
            prlinked_properties_extra_filters.append(date_filter)
            prlinked_properties_final_filters = " AND ".join(
                prlinked_properties_extra_filters
            )
        else:
            prlinked_properties_final_filters = date_filter

        prlinked_properties_selected_fields = [
            "id",
            "auction_code",
            "property_id",
            "person_id",
            "auction_date",
            "auction_price",
            "bidder_id",
            "date_entered",
            "entered_By",
            "businessdate",
        ]

        prlinked_properties = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "thesis_prlinked_properties",  # check name
            prlinked_properties_final_filters,
            prlinked_properties_selected_fields,
        )

        # HashKeys
        prlinked_properties_primary_key_fields = [
            prlinked_properties.id,
            lit("thesis"),
        ]

        # ================== prperson ==================

        prperson_extra_filters = None

        if prperson_extra_filters is not None:
            prperson_extra_filters.append(date_filter)
            prperson_final_filters = " AND ".join(prperson_extra_filters)
        else:
            prperson_final_filters = date_filter

        prperson_selected_fields = ["id", "afm", "last_name", "first_name"]

        prperson = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "thesis_prperson",  # check name
            prperson_final_filters,
            prperson_selected_fields,
        )

        # ================== Join ==================

        result_df_thesis = prlinked_properties.join(
            prperson,
            (prlinked_properties["person_id"] == prperson["id"]),
            "left",
        )

        result_df_thesis = result_df_thesis.withColumn(
            "AuctionRealEstateId",
            sha2(concat_ws(";", *prlinked_properties_primary_key_fields), 256),
        )

        # ================== Selection of Final Columns ==================

        result_df_thesis = result_df_thesis.select(
            result_df_thesis.AuctionRealEstateId.cast("string").alias(
                "AuctionRealEstateId"
            ),
            lit(None).alias("OriginSourceSystem"),
            lit("thesis").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).cast("string").alias("BatchId"),
            prlinked_properties.id.cast("string").alias("AuctionRealEstateKey"),
            prlinked_properties.auction_code.cast("string").alias(
                "AuctionRealEstateCode"
            ),
            lit(None).alias("AuctionCode"),
            lit(None).alias("AuctionObjKey"),
            lit(None).alias("RealEstateGroupCode"),
            lit(None).alias("PartyKey"),
            lit(None).alias("RealEstateKey"),
            lit(None).alias("RealEstateOwnershipKey"),
            lit(None).alias("CaseCode"),
            # prlinked_properties.property_id.cast("string").alias("RealEstateId"),
            prlinked_properties.person_id.cast("string").alias("DebtorPartyKey"),
            prperson.afm.cast("string").alias("OriginDebtorTaxRegistrationNumber"),
            concat_ws("-", prperson.last_name, prperson.first_name)
            .cast("string")
            .alias("OriginDebtorDescription"),
            prlinked_properties.auction_date.cast("Date").alias("AuctionDate"),
            prlinked_properties.auction_price.cast("string").alias("AuctionFinalPrice"),
            lit(None).alias("AuctionPublishedDate"),
            lit(None).alias("AuctionPostedDate"),
            lit(None).alias("AuctionFirstOfferPrice"),
            lit(None).alias("SeizureReportNumber"),
            lit(None).alias("SeizureReportYear"),
            lit(None).alias("HastenerCode"),
            lit(None).alias("Hastener"),
            lit(None).alias("HastenerTaxRegistrationNumber"),
            lit(None).alias("AuctionStatusCode"),
            lit(None).alias("AuctionStatus"),
            lit(None).alias("AuctioneerName"),
            lit(None).alias("AuctioneerEmail"),
            lit(None).alias("AuctioneerPhone"),
            lit(None).alias("TitleGR"),
            lit(None).alias("Area"),
            lit(None).alias("District"),
            lit(None).alias("Municipality"),
            lit(None).alias("Prefecture"),
            lit(None).alias("GeoCoordinatesX"),
            lit(None).alias("GeoCoordinatesY"),
            lit(None).alias("GeoLevel"),
            lit(None).alias("Address"),
            lit(None).alias("PostalCode"),
            lit(None).alias("Category"),
            lit(None).alias("SubCategory"),
            lit(None).alias("LandeaUrl"),
            lit(None).alias("FloorRawValue"),
            lit(None).alias("ConstructionYear"),
            lit(None).alias("SurfaceSqm"),
            lit(None).alias("Tags"),
            lit(None).alias("TagValues"),
            lit(None).alias("ActiveFlag"),
            lit(None).alias("TNMergeCode"),
            lit(None).alias("LandRegistration"),
            prlinked_properties.bidder_id.cast("string").alias("BidderCode"),
            prlinked_properties.date_entered.cast("string").alias("DateEntered"),
            prlinked_properties.entered_By.cast("string").alias("EnteredBy"),
            lit(None).alias("AuctionOwnershipTypeCode"),
            lit(None).alias("AuctionOwnershipType"),
            lit(None).alias("FirstBidder"),
            lit(None).alias("LawFirm"),
            lit(None).alias("FileDate"),
            lit(None).alias("FileName"),
            prlinked_properties.businessdate.cast("date").alias("BusinessDate"),
        )

        # ================== Append ==================

        result_df_thesis.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:

# COMMAND ----------

load_curated_auction_real_estate_thesis(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)